### Type Annotation Handling

In [8]:
from __future__ import annotations

### Implement each half (a-n, o-z, A-M, N-Z, 0-9) as its own closed cyclic range
- so a shift can never push a character out of its own category.
- to guarantee that encrypt_file() and decrypt_file() are always exact inverses of one another, no matter which non-negative integers are supplied for shift1 and shift2.

In [9]:
def _shift_within_range(char: str, start: str, size: int, amount: int) -> str:
    """Cyclically shift `char` by `amount` positions within a contiguous
    range of `size` characters that begins at `start`.

    A positive `amount` shifts forward (later in the alphabet/digits);
    a negative `amount` shifts backward.
    """
    offset = ord(char) - ord(start)
    new_offset = (offset + amount) % size
    return chr(ord(start) + new_offset)

### Shift rules (applied per character, based on the character's category):
- Lowercase a-n : shift forward within {a..n} by (shift1 * shift2)
- Lowercase o-z : shift backward within {o..z} by (shift1 + shift2)
- Uppercase A-M : shift backward within {A..M} by shift1
- Uppercase N-Z : shift forward within {N..Z} by shift2 ** 2
- Digits   0-9  : shift forward within {0..9} by (shift1 - shift2)
- Anything else : left unchanged (spaces, tabs, newlines, punctuation, symbols)

In [10]:

def _transform_char(char: str, shift1: int, shift2: int, *, encrypting: bool) -> str:
    """Apply the cipher rule for a single character.

    The same rule table is used for both directions: `encrypting=True`
    applies the forward (encryption) shift amounts, `encrypting=False`
    applies the exact negation of those amounts, which undoes them.
    """
    sign = 1 if encrypting else -1

    if 'a' <= char <= 'n':
        return _shift_within_range(char, 'a', 14, sign * (shift1 * shift2))
    if 'o' <= char <= 'z':
        return _shift_within_range(char, 'o', 12, -sign * (shift1 + shift2))
    if 'A' <= char <= 'M':
        return _shift_within_range(char, 'A', 13, -sign * shift1)
    if 'N' <= char <= 'Z':
        return _shift_within_range(char, 'N', 13, sign * (shift2 ** 2))
    if char.isdigit():
        return _shift_within_range(char, '0', 10, sign * (shift1 - shift2))

    return char

### encrypt_file(): read the plaintext, shift every character forward, and save the result
- reads the whole file at input_path as a single string.
- passes each character to _transform_char() with `encrypting=True`, applying the forward shift amounts.
- writes the encrypted text to output_path (e.g. `encrypted_text.txt`).

In [11]:
def encrypt_file(shift1: int, shift2: int, input_path: str, output_path: str) -> None:
    """Read the text at input_path, encrypt every character, and write the
    result to output_path.

    Every character is passed to _transform_char with encrypting=True, so the
    forward shift amounts are applied. Characters that are not letters or
    digits are returned unchanged, so spaces and punctuation are preserved.
    """
    f = open(input_path, 'r')
    text = f.read()
    f.close()

    encrypted = ""
    for char in text:
        encrypted += _transform_char(char, shift1, shift2, encrypting=True)

    f = open(output_path, 'w')
    f.write(encrypted)
    f.close()

### decrypt_file(): read the encrypted text and undo the shift on every character
- reads the whole file at input_path as a single string.
- passes each character to _transform_char() with `encrypting=False`, which applies the exact negation of the encryption shift.
- writes the recovered text to output_path (e.g. `decrypted_text.txt`).

In [12]:
def decrypt_file(shift1: int, shift2: int, input_path: str, output_path: str) -> None:
    """Read the encrypted text at input_path, decrypt every character, and
    write the result to output_path.

    Every character is passed to _transform_char with encrypting=False, which
    applies the exact negation of the encryption shift and therefore undoes it.
    """
    f = open(input_path, 'r')
    text = f.read()
    f.close()

    decrypted = ""
    for char in text:
        decrypted += _transform_char(char, shift1, shift2, encrypting=False)

    f = open(output_path, 'w')
    f.write(decrypted)
    f.close()

### Quick check: encrypt then decrypt raw_text.txt and confirm the original is recovered
- `shift1=2, shift2=1` is chosen deliberately: without the closed cyclic ranges, `n` and `s` would both map to `p`.
- prints `True` when the decrypted text is identical to the original (the official `verify_files()` is a separate function).

In [13]:
raw_path = "../Assignment 2 Description/raw_text.txt"

encrypt_file(2, 1, raw_path, "encrypted_text.txt")
decrypt_file(2, 1, "encrypted_text.txt", "decrypted_text.txt")

f = open(raw_path, 'r'); original = f.read(); f.close()
f = open("decrypted_text.txt", 'r'); restored = f.read(); f.close()
print("Round-trip OK:", original == restored)

Round-trip OK: True
